# Étape 6 : règles de nettoyage

Ce notebook applique aux données chargées des règles de nettoyage explicites, dans un ordre fixe, sans jamais modifier les fichiers bruts : il travaille sur une copie, et chaque ligne garde son lignage. Chaque modification est inscrite dans un journal des corrections (table, ligne source, colonne, règle, valeur avant et après). Une valeur que la règle ne sait pas corriger sans ambiguïté reste telle quelle et reste une anomalie. Les valeurs des colonnes sensibles ne sont jamais affichées.

Chaque mois : mettre à jour `PERIODE_FIN`, exécuter d'abord le notebook `02c_dictionnaire`, puis celui-ci en entier.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from frigeo import racine_projet
from frigeo.perimetre import decouvrir

# PARAMETRE A MODIFIER CHAQUE MOIS
# Dernier mois déposé dans data/raw, au format AAAA-MM
PERIODE_FIN = "2026-03"

In [ ]:
%%time
# CHARGEMENT

from frigeo.chargement import charger_tout
from frigeo.controles import charger_dictionnaire_enregistre
from frigeo.dictionnaire import charger_meta

DATA = racine_projet() / "data"
perimetre = decouvrir(DATA, PERIODE_FIN)
perimetre.verifier()
print(perimetre.resume())

tables = charger_tout(perimetre)
# Sans dictionnaire enregistré, le message renvoie au notebook 02c_dictionnaire.
dictionnaire = charger_dictionnaire_enregistre()
print(len(tables), "tables,", sum(len(c) for c in dictionnaire.values()), "colonnes au dictionnaire")
print("dictionnaire de la période", charger_meta()["periode_fin"], ", données jusqu'à", PERIODE_FIN)

In [ ]:
# REGLES DE NETTOYAGE

from frigeo.nettoyage import catalogue_nettoyage

# Règles appliquées, dans leur ordre, avec ce qu'elles corrigent et ce qu'elles ne
# corrigent jamais.
display(catalogue_nettoyage())

In [ ]:
%%time
# NETTOYAGE

from frigeo.nettoyage import ACTION_MODIFIEE, ACTION_RETIREE, nettoyer

# Les tables chargées ne sont pas modifiées : tables_nettoyees en est une copie
# corrigée. Le journal porte des valeurs du client, il n'est pas affiché.
# La date de coupure (fin de la période extraite) borne les années écrites sur deux chiffres.
tables_nettoyees, journal, bilan_nettoyage = nettoyer(
    tables, dictionnaire, date_coupure=perimetre.date_coupure
)
print(int((journal["action"] == ACTION_MODIFIEE).sum()), "cellules corrigées,",
      int((journal["action"] == ACTION_RETIREE).sum()), "lignes retirées,",
      int(bilan_nettoyage["non_corrigees"].sum()), "cellules ou lignes visées non corrigées")
display(bilan_nettoyage)

In [ ]:
# COLONNES SENSIBLES

from frigeo.confidentialite import charger_sensibilite, est_sensible
from frigeo.controles import VALEUR_MASQUEE
from frigeo.dictionnaire import est_masquee
from frigeo.doublons import charger_declarations_doublons

sensibles = charger_sensibilite()
declarations = charger_declarations_doublons()


def sensible(table, colonne):
    regles = dictionnaire[table][colonne]
    return (est_sensible(sensibles, table, colonne) or est_masquee(regles["valeurs"])
            or est_masquee(regles["format"]))


# Les valeurs ne figurent que sur les cellules modifiées (une ligne retirée n'en a pas).
modifiees = journal[journal["action"] == ACTION_MODIFIEE]
sur_sensible = [sensible(t, c) for t, c in zip(modifiees["table"], modifiees["colonne"])]
cle_sensible = [
    any(est_sensible(sensibles, t, c) for c in declarations.get(t, {}).get("cle", []))
    for t in journal["table"]
]
masquees = ((modifiees["valeur_avant"] == VALEUR_MASQUEE)
            & (modifiees["valeur_apres"] == VALEUR_MASQUEE))
print("corrections sur une colonne sensible", sum(sur_sensible),
      ", dont valeurs non masquées (0 attendu)", int((~masquees[sur_sensible]).sum()))
print("lignes du journal dont la clé a une colonne sensible", sum(cle_sensible),
      ", dont clés non masquées (0 attendu)",
      int((journal.loc[cle_sensible, "cle"] != VALEUR_MASQUEE).sum()))